##### Notebook testes
###### Este Notebook é apenas para testes e não é o projeto final.

## 1. Imports

Nesta secção são importadas todas as bibliotecas necessárias para o desenvolvimento do projeto. Incluem-se ferramentas para manipulação de dados, análise estatística, pré-processamento, criação de pipelines e preparação das fases de treino e avaliação dos modelos.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.cluster import KMeans

from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, silhouette_score
)

## 2. Load e limpeza primária

Nesta fase é feito o carregamento do dataset, a validação da variável-alvo escolhida e a remoção de colunas sem utilidade preditiva, como identificadores. Em seguida, é realizada uma limpeza primária com o objetivo de inspecionar os dados e documentar o seu estado antes e depois do tratamento dos valores em falta.

Esta etapa tem um propósito sobretudo exploratório e descritivo. Para efeitos de modelação, o pré-processamento final será posteriormente integrado num pipeline reprodutível, evitando transformações inconsistentes entre treino e teste.

In [2]:
# =========================
# CONFIGURAÇÃO
# =========================
link = r"Dados_Projeto\student_records_missing.csv"
allowed_cols = ["focus_index", "burnout_level", "productivity_score", "exam_score"]

# Tema escolhido:
# Regressão -> focus_index
# Classificação -> exam_score binário
target_reg = "focus_index"
target_clf_base = "exam_score"


# =========================
# FUNÇÕES AUXILIARES
# =========================
def summarize_numeric(df):
    #nomes das colunas numéricas
    num_cols = df.select_dtypes(include=np.number).columns.tolist()

    if not num_cols:
        return pd.DataFrame()

    summary = df[num_cols].agg(["count", "mean", "std", "min", "median", "max"]).T
    #valores em falta por coluna
    summary["missing"] = df[num_cols].isna().sum()
    summary = summary[["count", "missing", "mean", "std", "min", "median", "max"]]

    return summary.round(3)


def summarize_categorical(df):
    #nomes das colunas categóricas
    cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
    rows = []

    for col in cat_cols:
        s = df[col]
        mode = s.mode(dropna=True)

        rows.append({
            "variable": col,
            "count": s.notna().sum(),
            "missing": s.isna().sum(),
            "unique": s.nunique(dropna=True),
            "mode": mode.iloc[0] if not mode.empty else None
        })

    if rows:
        return pd.DataFrame(rows).set_index("variable")
    return pd.DataFrame()


def print_full_summary(title, df):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    print(f"\nShape: {df.shape}")

    print("\nTipos de dados:")
    print(df.dtypes)

    print("\nValores em falta por coluna:")
    print(df.isna().sum())

    print("\n--- Resumo Numérico ---")
    num_summary = summarize_numeric(df)
    if not num_summary.empty:
        print(num_summary)
    else:
        print("Sem colunas numéricas.")

    print("\n--- Resumo Categórico ---")
    cat_summary = summarize_categorical(df)
    if not cat_summary.empty:
        print(cat_summary)
    else:
        print("Sem colunas categóricas.")


def clean_for_report(X):
    """
    Limpeza primária apenas para análise descritiva.
    Não usar esta saída para treinar modelos se fores usar pipeline.
    """
    X_clean = X.copy()

    num_cols = X_clean.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_clean.select_dtypes(exclude=np.number).columns.tolist()

    for col in num_cols:
        #Mediana é usada nas numéricas porque costuma ser mais robusta a valores extremos.
        X_clean[col] = X_clean[col].fillna(X_clean[col].median())

    for col in cat_cols:
        mode = X_clean[col].mode(dropna=True)
        if not mode.empty:
            #A moda é usada nas categóricas porque representa a categoria mais comum.
            X_clean[col] = X_clean[col].fillna(mode.iloc[0])

    return X_clean


def load_project_data(fname, target_reg, target_clf_base):
    df = pd.read_csv(fname)

    for target in [target_reg, target_clf_base]:
        if target not in allowed_cols:
            raise ValueError(f"Target inválido: {target}. Targets permitidos: {allowed_cols}")

        if target not in df.columns:
            raise ValueError(f"Target '{target}' não encontrado no dataset.")

    # Criar target binário da classificação
    #1 - acima da mediana, 0 - abaixo da mediana (equilibra melhor as classes)
    threshold = df[target_clf_base].median()
    df["high_exam_score"] = (df[target_clf_base] >= threshold).astype(int)

    # Remover ID e targets que não entram como feature
    cols_to_drop = [
        "student_id",
        "focus_index",
        "exam_score",
        "high_exam_score",
        "burnout_level",
        "productivity_score"
    ]

    X_raw = df.drop(columns=cols_to_drop, errors="ignore").copy()
    y_reg = df[target_reg].copy()
    y_clf = df["high_exam_score"].copy()

    return df, X_raw, y_reg, y_clf, threshold


# =========================
# EXECUÇÃO
# =========================
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df, X_raw, y_reg, y_clf, exam_threshold = load_project_data(link, target_reg, target_clf_base)

print("Target da regressão:", target_reg)
print("Target da classificação: high_exam_score")
print("Threshold usado para high_exam_score:", round(exam_threshold, 3))

print("\nDistribuição da variável de classificação:")
print(y_clf.value_counts())

# ANTES
print_full_summary("VALORES ANTES (features sem limpeza primária)", X_raw)

# Limpeza primária apenas para relatório / inspeção
X_report_clean = clean_for_report(X_raw)

# DEPOIS
print_full_summary("VALORES DEPOIS (features após limpeza primária)", X_report_clean)

print("\n" + "=" * 90)
print("RESUMO FINAL DA LIMPEZA PRIMÁRIA")
print("=" * 90)
print("Número total de features:", X_raw.shape[1])
print("Missing values antes:", X_raw.isna().sum().sum())
print("Missing values depois:", X_report_clean.isna().sum().sum())

print("\nPrimeiras linhas após limpeza primária:")
print(X_report_clean.head())

Target da regressão: focus_index
Target da classificação: high_exam_score
Threshold usado para high_exam_score: 18.01

Distribuição da variável de classificação:
high_exam_score
1    2501
0    2499
Name: count, dtype: int64

VALORES ANTES (features sem limpeza primária)

Shape: (5000, 16)

Tipos de dados:
age                       int64
gender                   object
academic_level           object
study_hours             float64
self_study_hours        float64
online_classes_hours    float64
social_media_hours      float64
gaming_hours            float64
sleep_hours             float64
screen_time_hours       float64
exercise_minutes          int64
caffeine_intake_mg      float64
part_time_job             int64
upcoming_deadline         int64
internet_quality         object
mental_health_score     float64
dtype: object

Valores em falta por coluna:
age                       0
gender                  150
academic_level          100
study_hours               0
self_study_hours         

## 3. Pipeline de pré-processamento

Após a limpeza primária, foi construído um pipeline de pré-processamento para lidar de forma reprodutível com variáveis numéricas e categóricas. Esta opção evita repetir transformações manualmente e garante que o mesmo tratamento é aplicado de forma consistente aos dados de treino e de teste.

Nas variáveis numéricas, foi utilizada imputação pela mediana e normalização com `RobustScaler`. Nas variáveis categóricas, foi utilizada imputação pela moda e codificação com `OneHotEncoder`. Esta separação foi implementada com recurso a `Pipeline` e `ColumnTransformer`.

In [ ]:
# =========================
# PIPELINE DE PRÉ-PROCESSAMENTO
# =========================

numeric_features = X_raw.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_raw.select_dtypes(exclude=np.number).columns.tolist()

print("Colunas numéricas:")
print(numeric_features)

print("\nColunas categóricas:")
print(categorical_features)

# Pipeline numérico
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    # RobustScaler centra os dados na mediana (=0) e escala usando o IQR, ficando mais robusto a outliers
    # o que é consistente com a imputação pela mediana e com a natureza
    # potencialmente assimétrica das variáveis do dataset
    
    ("scaler", RobustScaler())
])

# Pipeline categórico
categorical_transformer = Pipeline(steps=[
    #preenche valores em falta com a categoria mais frequente
    ("imputer", SimpleImputer(strategy="most_frequent")),
    #transforma categorias em variáveis dummy (0/1) e ignora categorias desconhecidas no teste
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocessador final
preprocessor = ColumnTransformer(
    transformers=[
        #(nome, pipeline, aplicar a estas colunas)
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Teste do pipeline

#fit(aprende medianas-> missing, medias e desvios-> scaling, categorias-> onehot) + transform(aplica aos dados)
X_processed = preprocessor.fit_transform(X_raw)

print("\nShape antes do pipeline:", X_raw.shape)
print("Shape depois do pipeline:", X_processed.shape) #aumenta num columns; onehot expande colunas categóricas em várias colunas dummy

feature_names_out = preprocessor.get_feature_names_out()

print(f"\n{len(feature_names_out)} Features transformadas:")
print(feature_names_out)

Colunas numéricas:
['age', 'study_hours', 'self_study_hours', 'online_classes_hours', 'social_media_hours', 'gaming_hours', 'sleep_hours', 'screen_time_hours', 'exercise_minutes', 'caffeine_intake_mg', 'part_time_job', 'upcoming_deadline', 'mental_health_score']

Colunas categóricas:
['gender', 'academic_level', 'internet_quality']

Shape antes do pipeline: (5000, 16)
Shape depois do pipeline: (5000, 22)

22 Features transformadas:
['num__age' 'num__study_hours' 'num__self_study_hours'
 'num__online_classes_hours' 'num__social_media_hours' 'num__gaming_hours'
 'num__sleep_hours' 'num__screen_time_hours' 'num__exercise_minutes'
 'num__caffeine_intake_mg' 'num__part_time_job' 'num__upcoming_deadline'
 'num__mental_health_score' 'cat__gender_Female' 'cat__gender_Male'
 'cat__gender_Other' 'cat__academic_level_High School'
 'cat__academic_level_Postgraduate' 'cat__academic_level_Undergraduate'
 'cat__internet_quality_Average' 'cat__internet_quality_Good'
 'cat__internet_quality_Poor']


## 4. Modelos baseline

Nesta fase foram selecionados modelos baseline simples e interpretáveis, com o objetivo de estabelecer uma referência inicial de desempenho. Estes modelos permitem avaliar o comportamento dos dados antes da utilização de abordagens mais complexas.

Para a tarefa de regressão foi utilizada `LinearRegression` como modelo principal e `DecisionTreeRegressor` como modelo de comparação. Para a tarefa de classificação foi utilizada `LogisticRegression` como modelo principal e `DecisionTreeClassifier` como modelo de comparação.


## 5. Métricas de avaliação

Foram definidas métricas adequadas a cada tipo de problema, de forma a avaliar corretamente o desempenho dos modelos.

Na regressão foi utilizada como métrica principal o `RMSE` e como métrica secundária o `R²`. Na classificação foi utilizado o `F1-score` como métrica principal e a `Accuracy` como métrica secundária.

Estas métricas permitem analisar tanto o erro dos modelos como a sua capacidade de generalização.



## 6. Treino e avaliação dos modelos

Os modelos foram treinados utilizando pipelines que integram o pré-processamento definido anteriormente, garantindo consistência entre treino e teste.

Os dados foram divididos em conjuntos de treino e teste, e o desempenho dos modelos foi avaliado com base nas métricas definidas. Adicionalmente, foi utilizada validação cruzada para obter resultados mais robustos.